# Phase 3 — Incremental Improvement Ablation

**Workflow:**
1. Clone repo + install deps (reuse Phase 2 setup if already done)
2. Mount Drive + copy data
3. **[~5 min]** Quick sanity check on 5 frames
4. **[~2–3 h]** Full Phase 3 sweep — 6 configs × 30 frames
5. Print comparative table + Pareto curve
6. Save all results to Drive

**Phase 3 improvements tested (each isolated):**
- **3A — Objectness-aware loss**: anchor-level max-confidence suppression
- **3B — MI-Adam momentum**: gradient accumulation (μ=0.9)
- **3C — Multi-restart (n=3)**: random δ₀ inits (~3× compute)
- **3D — LPIPS + SSIM**: combined perceptual constraint
- **Combined**: all 4 improvements together

**Estimated total:** ~3 h (baseline+3A+3B+3D), ~9 h (if 3C/combined included).

## Cell P1 — Clone / pull repo

In [ ]:
import os

REPO_DIR = '/content/pfe-latent-attack'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    print('Repo already present — pulling ...')
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

for f in [
    'scripts/run_phase3_ablation.py',
    'configs/phase3_baseline.yaml',
    'configs/phase3_objectness.yaml',
    'configs/phase3_momentum.yaml',
    'configs/phase3_restart.yaml',
    'configs/phase3_ssim.yaml',
    'configs/phase3_combined.yaml',
    'src/losses.py',
    'src/attack.py',
]:
    status = chr(10003) + ' ' if os.path.exists(f) else chr(10007) + ' MISSING'
    print(f'  {status}  {f}')


## Cell P2 — Mount Drive and copy data

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
ASSETS = '/content/drive/MyDrive/pfe_assets'

os.makedirs('runs/yolov8n_detrac', exist_ok=True)
if not os.path.exists('runs/yolov8n_detrac/best.pt'):
    shutil.copy(f'{ASSETS}/best.pt', 'runs/yolov8n_detrac/best.pt')
print(f'{chr(10003)} best.pt ({os.path.getsize("runs/yolov8n_detrac/best.pt")//1024} KB)')

if not os.path.exists('data/images_50'):
    shutil.copytree(f'{ASSETS}/images_50', 'data/images_50')
n50 = len([f for f in os.listdir('data/images_50') if f.endswith('.jpg')])
print(f'{chr(10003)} data/images_50: {n50} frames')

os.makedirs('results/phase3', exist_ok=True)
print(f'{chr(10003)} Output directories ready')


## Cell P3 — Install dependencies

In [ ]:
!pip install lpips>=0.1.4 torchmetrics -q
!pip install -r requirements.txt -q

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

try:
    from torchmetrics.detection import MeanAveragePrecision
    print(chr(10003), 'torchmetrics.detection available')
except ImportError:
    print(chr(10007), 'torchmetrics manquant')

try:
    import lpips
    print(chr(10003), 'lpips available')
except ImportError:
    print(chr(10007), 'lpips manquant')


## Cell P4 — Sanity check: baseline on 5 frames (~3 min)

Runs `phase3_baseline` on **5 frames** to confirm the baseline DFR matches
the Phase 2 result. Expected: DFR ~ +0.20, DFR_strict ~ 0.50.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_phase3_ablation.py \
    --data      data/images_50 \
    --n_frames  5 \
    --output    results/phase3_sanity \
    --configs   configs/phase3_baseline.yaml \
    --no_resume


In [ ]:
import json

s = json.load(open('results/phase3_sanity/phase3_baseline/summary.json'))
print('Sanity baseline (5 frames):')
print(f"  DFR (prop)  = {s['mean_dfr']:+.4f}  +/- {s['se_dfr']:.4f}")
print(f"  DFR strict  = {s['dfr_strict_rate']:.3f}  "
      f"({int(s['dfr_strict_count'])}/{s['n_valid']} frames)")
print(f"  ASR         = {s['mean_asr']:.3f}")
print(f"  mAP drop    = {s['mean_map_drop']:.4f}")
print(f"  LPIPS       = {s['mean_lpips']:.4f}")
print()
print('Expected baseline: DFR ~ +0.20, DFR_strict ~ 0.50')


## Cell P5 — Full Phase 3 sweep — no restarts (~2.5 h)

Runs **baseline + 3A + 3B + 3D** on 30 frames.
Excludes `phase3_restart` and `phase3_combined` to save compute.
Run those in Cell P5b if time permits.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_phase3_ablation.py \
    --data      data/images_50 \
    --n_frames  30 \
    --output    results/phase3 \
    --configs   configs/phase3_baseline.yaml \
                configs/phase3_objectness.yaml \
                configs/phase3_momentum.yaml \
                configs/phase3_ssim.yaml \
    --resume


## Cell P5b — Restart + Combined configs (~6 h, optional)

Run only if you have enough Colab time.
`phase3_restart` runs 3× per frame; `phase3_combined` stacks all improvements (~9×).

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Uncomment to run — expensive!
# !python scripts/run_phase3_ablation.py \
#     --data      data/images_50 \
#     --n_frames  30 \
#     --output    results/phase3 \
#     --configs   configs/phase3_restart.yaml \
#                 configs/phase3_combined.yaml \
#     --resume

print('Cell P5b skipped. Uncomment lines above to run restart/combined configs.')


## Cell P6 — Comparative results table

In [ ]:
import json
from pathlib import Path

results_dir = Path('results/phase3')
summaries = {}
for cfg_dir in sorted(results_dir.iterdir()):
    summary_f = cfg_dir / 'summary.json'
    if summary_f.exists():
        summaries[cfg_dir.name] = json.load(open(summary_f))

header = f"{'Config':<30} {'DFR':>8} {'DFR_str':>9} {'ASR':>7} {'mAP_drop':>10} {'LPIPS':>8} {'Steps':>7}"
print(header)
print('-' * len(header))

for tag, s in summaries.items():
    dfr   = f"{s['mean_dfr']:+.4f}"        if s.get('mean_dfr') is not None else '   N/A'
    dstr  = f"{s['dfr_strict_rate']:.3f}"   if s.get('dfr_strict_rate') is not None else '    N/A'
    asr   = f"{s['mean_asr']:.3f}"           if s.get('mean_asr') is not None else '   N/A'
    mapd  = f"{s['mean_map_drop']:.4f}"      if s.get('mean_map_drop') is not None else '     N/A'
    lpips = f"{s['mean_lpips']:.4f}"          if s.get('mean_lpips') is not None else '    N/A'
    steps = f"{s.get('mean_steps', 0):.1f}"
    print(f"{tag:<30} {dfr:>8} {dstr:>9} {asr:>7} {mapd:>10} {lpips:>8} {steps:>7}")

print('\nLegende: DFR_str=fraction frames n_adv==0 | ASR=toutes classes supprimees')


In [ ]:
md_table = Path('results/phase3/phase3_table.md')
if md_table.exists():
    print(open(md_table).read())
else:
    print('Run Cell P5 first to generate the table.')


## Cell P7 — Phase 3 Pareto curve (DFR_strict vs LPIPS)

In [ ]:
import json, math
from pathlib import Path
import matplotlib.pyplot as plt

results_dir = Path('results/phase3')
summaries = {}
for cfg_dir in sorted(results_dir.iterdir()):
    summary_f = cfg_dir / 'summary.json'
    if summary_f.exists():
        summaries[cfg_dir.name] = json.load(open(summary_f))

COLORS = {
    'phase3_baseline':   '#2196F3',
    'phase3_objectness': '#4CAF50',
    'phase3_momentum':   '#FF9800',
    'phase3_ssim':       '#9C27B0',
    'phase3_restart':    '#F44336',
    'phase3_combined':   '#000000',
}
LABELS = {
    'phase3_baseline':   'Baseline',
    'phase3_objectness': '+ Objectness (3A)',
    'phase3_momentum':   '+ Momentum (3B)',
    'phase3_ssim':       '+ LPIPS+SSIM (3D)',
    'phase3_restart':    '+ Restart (3C)',
    'phase3_combined':   'Combined',
}

fig, ax = plt.subplots(figsize=(8, 6))
for tag, s in summaries.items():
    lpips_v = s.get('mean_lpips')
    dfr_v   = s.get('dfr_strict_rate')
    if lpips_v is None or dfr_v is None:
        continue
    if not (math.isfinite(lpips_v) and math.isfinite(dfr_v)):
        continue
    color = COLORS.get(tag, '#888888')
    label = LABELS.get(tag, tag)
    ax.scatter(lpips_v, dfr_v, s=140, color=color, zorder=5, label=label)
    ax.annotate(label, (lpips_v, dfr_v),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, color=color)

ax.set_xlabel('LPIPS down (lower = less distortion)', fontsize=11)
ax.set_ylabel('DFR_strict up (fraction frames fully suppressed)', fontsize=11)
ax.set_title('Phase 3 Pareto: Effectiveness vs. Perceptual Quality', fontsize=12)
ax.legend(loc='lower left', fontsize=8, framealpha=0.7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase3/phase3_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/phase3/phase3_pareto.png')


## Cell P8 — Save all Phase 3 results to Drive

In [ ]:
import shutil, os, time
from pathlib import Path

ASSETS = '/content/drive/MyDrive/pfe_assets'
local_dir = 'results/phase3'
drive_dst = f'{ASSETS}/phase3'

# Backup si la destination existe déjà
if os.path.exists(drive_dst):
    backup = f'{drive_dst}_backup_{int(time.time())}'
    shutil.move(drive_dst, backup)
    print(f'⚠  Backup ancien run → {Path(backup).name}')

shutil.copytree(local_dir, drive_dst)
print('✓ results/phase3 → Drive/pfe_assets/phase3')
print('\nDone. Retrieve from pfe_assets/phase3/phase3_summary.json')
